In [14]:
# ============ CELL 1 — SETUP ============
!pip install catboost -q

from google.colab import drive
drive.mount('/content/drive')

import gc, os, time, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import psutil

from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb, xgboost as xgb
from catboost import CatBoostRegressor
import joblib

def ram(): return f"{psutil.Process().memory_info().rss/1e9:.2f} GB"
def tick(t0): return f"{time.time()-t0:.1f}s"

print(f"✅ Setup | RAM: {ram()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup | RAM: 2.11 GB


In [15]:
# ============ CELL 2 — LOAD (CHUNKED) ============
t0 = time.time()

CANDIDATES = ["/content/drive/MyDrive/last_2_years.csv",
              "/content/drive/MyDrive/data.csv"]
csv_path = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert csv_path, "❌ CSV not found"

# ⚡ RAM SETTING
ROWS_TO_KEEP = 5_000_000    # 50L rows enough with lags; 0 = full
CHUNK_SIZE   = 500_000

dtype_map = {"State":"category","District":"category","Market":"category",
             "Commodity":"category","Variety":"category",
             "Min_Price":"float32","Max_Price":"float32","Modal_Price":"float32"}

chunks = []
total = 0
for chunk in pd.read_csv(csv_path, dtype=dtype_map, chunksize=CHUNK_SIZE,
                         low_memory=False):
    chunks.append(chunk)
    total += len(chunk)
    if ROWS_TO_KEEP and total >= ROWS_TO_KEEP:
        break
    gc.collect()

df = pd.concat(chunks, ignore_index=True)
del chunks; gc.collect()

# If over limit, sample
if ROWS_TO_KEEP and len(df) > ROWS_TO_KEEP:
    df = df.sample(n=ROWS_TO_KEEP, random_state=42).reset_index(drop=True)

print(f"✅ Loaded {len(df):,} rows in {tick(t0)} | RAM: {ram()}")

✅ Loaded 5,000,000 rows in 16.1s | RAM: 2.29 GB


In [16]:
# ============ CELL 3 — CLEAN ============
t0 = time.time()
price_cols = ["Min_Price","Max_Price","Modal_Price"]

# Date
df["date"] = pd.to_datetime(df["Arrival_Date"], dayfirst=True, errors="coerce")
if df["date"].isna().mean() > 0.2:
    df["date"] = pd.to_datetime(df["Arrival_Date"], errors="coerce")
df = df.dropna(subset=["date"]).drop(columns=["Arrival_Date"]).reset_index(drop=True)

# Time features (only what we need)
df["Month"]     = df["date"].dt.month.astype("int8")
df["DayOfWeek"] = df["date"].dt.dayofweek.astype("int8")
df["DayOfYear"] = df["date"].dt.dayofyear.astype("int16")

# Price clean
for c in price_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    df.loc[df[c] <= 0, c] = np.nan

row_med = df[price_cols].median(axis=1, skipna=True)
gmed = {c: df.loc[df[c] > 0, c].median() for c in price_cols}
for c in price_cols:
    m = df[c].isna()
    df.loc[m, c] = row_med[m]
    s = df[c].isna()
    df.loc[s, c] = gmed[c]
    df[c] = df[c].astype("float32")
del row_med; gc.collect()

# Fix Min/Max + clamp Modal
mask = df["Min_Price"] > df["Max_Price"]
df.loc[mask, ["Min_Price","Max_Price"]] = df.loc[mask, ["Max_Price","Min_Price"]].values
df["Modal_Price"] = df["Modal_Price"].clip(df["Min_Price"], df["Max_Price"]).astype("float32")

print(f"✅ Cleaned in {tick(t0)} | Rows: {len(df):,} | RAM: {ram()}")

✅ Cleaned in 8.0s | Rows: 5,000,000 | RAM: 2.31 GB


In [17]:
# ============ CELL 4 — DAILY AGG (KEY STEP) ============
t0 = time.time()
group_keys = ["State","District","Market","Commodity","Variety"]
price_cols = ["Min_Price","Max_Price","Modal_Price"]

# Sort + filter groups
df = df.sort_values(group_keys + ["date"]).reset_index(drop=True)

gcount = df.groupby(group_keys, observed=True).size()
valid = gcount[gcount >= 30].index
df = df.set_index(group_keys).loc[valid].reset_index()
del gcount, valid; gc.collect()
print(f"   Filtered groups ({tick(t0)}) | RAM: {ram()}")

# Aggregate
daily = (df.groupby(group_keys + ["date"], observed=True)[price_cols]
           .mean().reset_index()
           .sort_values(group_keys + ["date"]).reset_index(drop=True))

# 🔥🔥🔥 CRITICAL: Row-level df FREE karo (10x memory)
del df; gc.collect()

for c in price_cols:
    daily[c] = daily[c].astype("float32")

# Add date features to daily
daily["Month"]     = daily["date"].dt.month.astype("int8")
daily["DayOfWeek"] = daily["date"].dt.dayofweek.astype("int8")
daily["DayOfYear"] = daily["date"].dt.dayofyear.astype("int16")

print(f"✅ Daily agg in {tick(t0)}")
print(f"   Daily rows: {len(daily):,}")
print(f"💾 RAM: {ram()}")

   Filtered groups (18.8s) | RAM: 2.29 GB
✅ Daily agg in 26.7s
   Daily rows: 4,734,873
💾 RAM: 2.22 GB


In [18]:
# ============ CELL 5 — LAGS ============
t0 = time.time()
gk = group_keys

g = daily.groupby(gk, observed=True, sort=False)
for lag in [1, 2, 3, 5, 7, 14, 21, 30, 60, 90]:
    daily[f"Modal_lag_{lag}"] = g["Modal_Price"].shift(lag).astype("float32")
for lag in [1, 3, 7, 14]:
    daily[f"Min_lag_{lag}"] = g["Min_Price"].shift(lag).astype("float32")
    daily[f"Max_lag_{lag}"] = g["Max_Price"].shift(lag).astype("float32")
del g; gc.collect()

print(f"✅ Lags in {tick(t0)} | Cols: {daily.shape[1]} | RAM: {ram()}")

✅ Lags in 3.3s | Cols: 30 | RAM: 2.54 GB


In [19]:
# ============ CELL 6 — ROLLING ============
t0 = time.time()
gk_idx = list(range(len(group_keys)))

daily["_s1"] = daily.groupby(group_keys, observed=True)["Modal_Price"].shift(1).astype("float32")

def groll(col, w, method, mp=1):
    r = daily.groupby(group_keys, observed=True)[col].rolling(w, min_periods=mp).agg(method)
    r.index = r.index.droplevel(gk_idx)
    return r.sort_index()

for w in [7, 14, 30]:
    daily[f"Modal_roll_mean_{w}"] = groll("_s1", w, "mean", 1).astype("float32")
    daily[f"Modal_roll_std_{w}"]  = groll("_s1", w, "std",  2).astype("float32")
for w in [7, 30]:
    daily[f"Modal_roll_min_{w}"] = groll("_s1", w, "min", 1).astype("float32")
    daily[f"Modal_roll_max_{w}"] = groll("_s1", w, "max", 1).astype("float32")

daily["Modal_trend_7"]  = (daily["Modal_lag_1"] - daily["Modal_lag_7"]).astype("float32")
daily["Modal_trend_30"] = (daily["Modal_lag_1"] - daily["Modal_lag_30"]).astype("float32")
daily["Modal_volatility_7"]  = (daily["Modal_roll_std_7"]  / (daily["Modal_roll_mean_7"]  + 1)).astype("float32")
daily["Min_Max_ratio_lag1"]  = (daily["Min_lag_1"] / (daily["Max_lag_1"] + 1)).astype("float32")
daily["Modal_Max_ratio_lag1"]= (daily["Modal_lag_1"] / (daily["Max_lag_1"] + 1)).astype("float32")

daily = daily.drop(columns=["_s1"])
gc.collect()
print(f"✅ Rolling in {tick(t0)} | Cols: {daily.shape[1]} | RAM: {ram()}")

✅ Rolling in 47.7s | Cols: 45 | RAM: 3.55 GB


In [20]:
# ============ CELL 7 — SEASONALITY + AGGREGATES ============
t0 = time.time()

# Cyclical
doy = daily["date"].dt.dayofyear
mon = daily["date"].dt.month
daily["sin_doy"]   = np.sin(2*np.pi*doy/365.25).astype("float32")
daily["cos_doy"]   = np.cos(2*np.pi*doy/365.25).astype("float32")
daily["sin_month"] = np.sin(2*np.pi*mon/12).astype("float32")
daily["cos_month"] = np.cos(2*np.pi*mon/12).astype("float32")
daily["is_weekend"]= (daily["DayOfWeek"] >= 5).astype("int8")

# Z-score
daily["Modal_zscore_30"] = ((daily["Modal_lag_1"] - daily["Modal_roll_mean_30"]) /
                            (daily["Modal_roll_std_30"] + 1)).astype("float32")
del doy, mon; gc.collect()

# Commodity-day
cd = daily.groupby(["Commodity","date"], observed=True)["Modal_Price"].agg(["mean","std"]).reset_index()
cd.columns = ["Commodity","date","comm_mean","comm_std"]
cd = cd.sort_values(["Commodity","date"])
cd["comm_mean_lag1"] = cd.groupby("Commodity")["comm_mean"].shift(1).astype("float32")
daily = daily.merge(cd[["Commodity","date","comm_mean_lag1"]], on=["Commodity","date"], how="left")
del cd; gc.collect()

# State-day
sd = daily.groupby(["State","date"], observed=True)["Modal_Price"].mean().reset_index()
sd.columns = ["State","date","state_mean"]
sd = sd.sort_values(["State","date"])
sd["state_mean_lag1"] = sd.groupby("State")["state_mean"].shift(1).astype("float32")
daily = daily.merge(sd[["State","date","state_mean_lag1"]], on=["State","date"], how="left")
del sd; gc.collect()

# Relative
daily["rel_to_comm"]  = (daily["Modal_lag_1"] / (daily["comm_mean_lag1"]  + 1)).astype("float32")
daily["rel_to_state"] = (daily["Modal_lag_1"] / (daily["state_mean_lag1"] + 1)).astype("float32")

# Force float32
for c in daily.columns:
    if daily[c].dtype == "float64":
        daily[c] = daily[c].astype("float32")
gc.collect()

print(f"✅ Seasonality in {tick(t0)} | Cols: {daily.shape[1]} | RAM: {ram()}")

✅ Seasonality in 6.3s | Cols: 55 | RAM: 3.13 GB


In [21]:
# ============ CELL 8 (FIXED) ============
t0 = time.time()
gk = group_keys
price_cols = ["Min_Price","Max_Price","Modal_Price"]

# Sort + split
daily = daily.sort_values("date").reset_index(drop=True)
split = int(len(daily) * 0.8)
print(f"   Train: {daily['date'].iloc[0].date()} → {daily['date'].iloc[split-1].date()}")
print(f"   Test : {daily['date'].iloc[split].date()} → {daily['date'].iloc[-1].date()}")

# ⚡ FIX: lag_cols me already Month/DayOfWeek/DayOfYear hain — duplicate mat karo
cat_features = ["State","District","Market","Commodity","Variety"]
lag_cols = [c for c in daily.columns if c not in gk + ["date"] + price_cols]
num_features = lag_cols   # ⬅️ duplicate hata diya
feature_cols = cat_features + num_features
target_cols = price_cols

# Sanity check — duplicate feature name?
assert len(feature_cols) == len(set(feature_cols)), \
       f"❌ Duplicate features: {[c for c in feature_cols if feature_cols.count(c) > 1]}"
print(f"   ✅ No duplicates | {len(feature_cols)} features")

X_train = daily.iloc[:split][feature_cols].copy()
y_train = daily.iloc[:split][target_cols].copy()
X_test  = daily.iloc[split:][feature_cols].copy()
y_test  = daily.iloc[split:][target_cols].copy()

del daily; gc.collect()

# Encode
for c in cat_features:
    cats = X_train[c].astype("category").cat.categories
    X_train[c] = pd.Categorical(X_train[c], categories=cats).codes.astype("int32")
    X_test[c]  = pd.Categorical(X_test[c],  categories=cats).codes.astype("int32")

# Log
y_train_log = np.log1p(y_train).astype("float32")
y_test_log  = np.log1p(y_test).astype("float32")

print(f"✅ Ready in {tick(t0)}")
print(f"   X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"   Features: {len(feature_cols)} | RAM: {ram()}")

   Train: 2024-09-13 → 2025-04-07
   Test : 2025-04-07 → 2025-05-27
   ✅ No duplicates | 51 features
✅ Ready in 7.2s
   X_train: (3787898, 51) | X_test: (946975, 51)
   Features: 51 | RAM: 2.04 GB


In [22]:
# ============ CELL 9 — EVAL ============
def evaluate(name, model, X, y_true, is_log=True):
    t = time.time()
    p = model.predict(X)
    if p.ndim == 1: p = p.reshape(-1, 3)
    if is_log: p = np.expm1(p)
    pdf = pd.DataFrame(p, columns=target_cols, index=y_true.index)
    rows = {c: {
        "MAE":  mean_absolute_error(y_true[c], pdf[c]),
        "RMSE": np.sqrt(mean_squared_error(y_true[c], pdf[c])),
        "R2":   r2_score(y_true[c], pdf[c]),
        "MAPE": np.mean(np.abs((y_true[c]-pdf[c]) /
                        np.maximum(np.abs(y_true[c]),1)))*100,
    } for c in target_cols}
    rep = pd.DataFrame(rows).T.round(3)
    print(f"\n{'='*55}\n📊 {name}  ({tick(t)})\n{'='*55}")
    print(rep)
    print(f"   Avg R²: {rep['R2'].mean():.4f} | Avg MAPE: {rep['MAPE'].mean():.2f}%")
    return pdf, rep

print("✅ Eval ready")

✅ Eval ready


In [23]:
# ============ CELL 10 — CATBOOST ============
t0 = time.time()
print("🚀 CatBoost training...")

cat_model = CatBoostRegressor(
    iterations=2000, learning_rate=0.03, depth=8,
    l2_leaf_reg=5.0, loss_function="MultiRMSE",
    random_seed=42, verbose=200,
    task_type="GPU",     # CPU ho to "CPU"
    early_stopping_rounds=100,
)
cat_model.fit(X_train, y_train_log,
              eval_set=(X_test, y_test_log), use_best_model=True)

print(f"⏱ CatBoost: {tick(t0)}")
y_pred_cat, rep_cat = evaluate("CatBoost", cat_model, X_test, y_test, is_log=True)
gc.collect(); print(f"💾 RAM: {ram()}")

🚀 CatBoost training...
0:	learn: 1.3764534	test: 1.4220291	best: 1.4220291 (0)	total: 259ms	remaining: 8m 36s
200:	learn: 0.2760075	test: 0.2533660	best: 0.2533660 (200)	total: 16.9s	remaining: 2m 30s
400:	learn: 0.2647810	test: 0.2465980	best: 0.2465980 (400)	total: 32.6s	remaining: 2m 10s
600:	learn: 0.2601866	test: 0.2440920	best: 0.2440920 (600)	total: 48.6s	remaining: 1m 53s
800:	learn: 0.2572219	test: 0.2427314	best: 0.2427314 (800)	total: 1m 4s	remaining: 1m 37s
1000:	learn: 0.2549771	test: 0.2418541	best: 0.2418541 (1000)	total: 1m 20s	remaining: 1m 20s
1200:	learn: 0.2531660	test: 0.2411718	best: 0.2411718 (1200)	total: 1m 36s	remaining: 1m 4s
1400:	learn: 0.2517638	test: 0.2406349	best: 0.2406349 (1400)	total: 1m 52s	remaining: 48.2s
1600:	learn: 0.2505612	test: 0.2402226	best: 0.2402226 (1600)	total: 2m 8s	remaining: 32s
1800:	learn: 0.2495506	test: 0.2398989	best: 0.2398989 (1800)	total: 2m 25s	remaining: 16s
1999:	learn: 0.2486391	test: 0.2396238	best: 0.2396238 (1999)	tot

In [28]:
# ============ CELL 12 (FINAL) — Naive + CV (CPU for CV) ============
# ⚡ NaN drop for naive comparison
valid_mask = X_test["Modal_lag_1"].notna() & y_test.notna().all(axis=1)
print(f"Valid test rows: {valid_mask.sum():,} / {len(X_test):,}")

X_test_v = X_test[valid_mask]
y_test_v = y_test[valid_mask]
y_pred_cat_v = y_pred_cat[valid_mask]

# --- Naive baseline ---
naive_pred = pd.DataFrame({c: X_test_v["Modal_lag_1"].values for c in target_cols},
                          index=y_test_v.index)
naive_rep = pd.DataFrame({c: {
    "MAE": mean_absolute_error(y_test_v[c], naive_pred[c]),
    "R2":  r2_score(y_test_v[c], naive_pred[c]),
    "MAPE": np.mean(np.abs((y_test_v[c]-naive_pred[c]) /
                    np.maximum(np.abs(y_test_v[c]),1)))*100}
    for c in target_cols}).T.round(3)

# --- CatBoost on valid rows ---
cat_rep_v = pd.DataFrame({c: {
    "MAE":  mean_absolute_error(y_test_v[c], y_pred_cat_v[c]),
    "R2":   r2_score(y_test_v[c], y_pred_cat_v[c]),
    "MAPE": np.mean(np.abs((y_test_v[c]-y_pred_cat_v[c]) /
                    np.maximum(np.abs(y_test_v[c]),1)))*100}
    for c in target_cols}).T.round(3)

board = pd.DataFrame({
    "CatBoost": cat_rep_v["R2"],
    "Naive":    naive_rep["R2"],
})
print("\n" + "="*60)
print("🏆 R² LEADERBOARD")
print("="*60)
print(board.round(4))
print("\n📊 AVERAGE R²:")
for i, (k, v) in enumerate(board.mean().sort_values(ascending=False).items(), 1):
    m = "🥇" if i==1 else "  "
    print(f"   {m} {k:<10}: {v:.4f}")

improvement = board["CatBoost"].mean() - board["Naive"].mean()
print(f"\n✅ CatBoost beats Naive by +{improvement:.4f} R²")

# --- CV (CPU — MultiRMSE GPU me nahi chalta) ---
print("\n🔄 CV (3-fold, CPU)...")
valid_train = X_train["Modal_lag_1"].notna().values
X_train_f = X_train[valid_train].reset_index(drop=True)
y_train_log_f = y_train_log[valid_train].reset_index(drop=True)
y_train_f = y_train[valid_train].reset_index(drop=True)

cv_n = min(150_000, len(X_train_f))
Xc = X_train_f.iloc[-cv_n:].reset_index(drop=True)
yc = y_train_log_f.iloc[-cv_n:].reset_index(drop=True)
yco = y_train_f.iloc[-cv_n:].reset_index(drop=True)
print(f"CV sample: {len(Xc):,} rows")

cv_r2, cv_mape = [], []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=3).split(Xc), 1):
    t0_cv = time.time()
    m = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=7,
        loss_function="MultiRMSE", random_seed=42,
        verbose=False,
        task_type="CPU",          # ⬅️ CPU — GPU MultiRMSE CV me fail hota hai
        thread_count=-1)
    m.fit(Xc.iloc[tr], yc.iloc[tr])
    p = np.expm1(m.predict(Xc.iloc[va]))
    if p.ndim == 1: p = p.reshape(-1, 3)
    yv = yco.iloc[va]
    r2s = [r2_score(yv[c], p[:, i]) for i, c in enumerate(target_cols)]
    mp  = [np.mean(np.abs((yv[c].values - p[:, i]) /
                  np.maximum(np.abs(yv[c].values), 1))) * 100
           for i, c in enumerate(target_cols)]
    cv_r2.append(np.mean(r2s)); cv_mape.append(np.mean(mp))
    print(f"   Fold {fold}: R²={np.mean(r2s):.4f} | MAPE={np.mean(mp):.2f}% | {tick(t0_cv)}")

print(f"\n🎯 CV Avg R²: {np.mean(cv_r2):.4f} | CV MAPE: {np.mean(cv_mape):.2f}%")
print(f"💾 RAM: {ram()}")

Valid test rows: 946,771 / 946,975

🏆 R² LEADERBOARD
             CatBoost  Naive
Min_Price       0.948  0.833
Max_Price       0.896  0.839
Modal_Price     0.900  0.855

📊 AVERAGE R²:
   🥇 CatBoost  : 0.9147
      Naive     : 0.8423

✅ CatBoost beats Naive by +0.0723 R²

🔄 CV (3-fold, CPU)...
CV sample: 150,000 rows
   Fold 1: R²=0.9234 | MAPE=8.71% | 42.4s
   Fold 2: R²=0.9670 | MAPE=8.16% | 51.6s
   Fold 3: R²=0.9619 | MAPE=7.64% | 61.2s

🎯 CV Avg R²: 0.9508 | CV MAPE: 8.17%
💾 RAM: 4.55 GB


In [29]:
# ============ CELL 13 — SAVE ============
SAVE_PATH = "/content/drive/MyDrive/catboost_final_kisan_setu_price_model.pkl"

joblib.dump({
    "model":        cat_model,
    "feature_cols": feature_cols,
    "cat_features": cat_features,
    "num_features": num_features,
    "target_cols":  target_cols,
    "is_log":       True,
    "model_name":   "CatBoost",
    "test_r2_avg":  rep_cat["R2"].mean(),
    "test_mape":    rep_cat["MAPE"].mean(),
    "cv_r2_avg":    float(np.mean(cv_r2)),
}, SAVE_PATH)

print(f"✅ SAVED: {SAVE_PATH}")
print(f"   Model    : CatBoost")
print(f"   Test R²  : {rep_cat['R2'].mean():.4f}")
print(f"   Test MAPE: {rep_cat['MAPE'].mean():.2f}%")
print(f"   CV R²    : {np.mean(cv_r2):.4f}")
print(f"   Size     : {os.path.getsize(SAVE_PATH)/1e6:.1f} MB")

✅ SAVED: /content/drive/MyDrive/catboost_final_kisan_setu_price_model.pkl
   Model    : CatBoost
   Test R²  : 0.9147
   Test MAPE: 7.62%
   CV R²    : 0.9508
   Size     : 42.3 MB
